In [12]:
import pandas as pd
import numpy as np
import os

# 1. SET PATHS
climate_path = "../data/raw/mp_15yr_climate_backbone.csv"
ndvi_path = "../data/raw/MP_NDVI_2010_2024_Master.csv" # Your GEE CSV

# 2. LOAD DATASETS
print("Loading NASA Climate Backbone and GEE NDVI data...")
df_climate = pd.read_csv(climate_path)
df_ndvi = pd.read_csv(ndvi_path)

# 3. STANDARDIZE COLUMN NAMES, DATES, AND TYPES
df_climate['Date'] = pd.to_datetime(df_climate['Date'], format='%Y%m%d') 
df_ndvi['Date'] = pd.to_datetime(df_ndvi['Date']) 

print(f"Date Range fixed: {df_climate['Date'].min()} to {df_climate['Date'].max()}") 

# Align GEE column names (mean -> NDVI, district_name -> District)
name_col = 'district_name' if 'district_name' in df_ndvi.columns else 'district'
df_ndvi = df_ndvi.rename(columns={name_col: 'District', 'mean': 'NDVI'})

# --- THE FIX: Convert both District columns to strings (Object) ---
df_climate['District'] = df_climate['District'].astype(str).str.strip()
df_ndvi['District'] = df_ndvi['District'].astype(str).str.strip()

# 4. THE CORRECTED MERGE
print("Merging layers into National Stack Master Table...")
master = pd.merge(
    df_climate, 
    df_ndvi[['Date', 'District', 'NDVI']], 
    on=['Date', 'District'], 
    how='outer'
)

# Re-sort
master = master.sort_values(['District', 'Date']).reset_index(drop=True) 
print(f"✅ Merge successful. Rows: {len(master)}")

# 5. DATA ENGINEERING: GAP FILLING (Interpolation)
# Filling the 16-day NDVI gaps to create a daily continuous stream
print("Interpolating 16-day NDVI gaps to daily frequency...")
master['NDVI'] = master.groupby('District')['NDVI'].transform(
    lambda x: x.interpolate(method='linear').ffill().bfill()
)

# 6. FEATURE ENGINEERING (Q14 & Q15 Implementation)
print("Computing climatological anomalies and drought indices...")

# A. District-specific 30-year thresholds (Climatology Proxy)
master['T_95th'] = master.groupby('District')['LST'].transform(lambda x: x.quantile(0.95))
master['SM_5th'] = master.groupby('District')['SSM'].transform(lambda x: x.quantile(0.05))

# B. Anomaly Indicator Flags
master['Heat_Anomaly'] = (master['LST'] >= master['T_95th']).astype(int)
master['Drought_Anomaly'] = (master['SSM'] <= master['SM_5th']).astype(int)

# C. SPI Proxy (Q15 Requirement: Precipitation Anomaly)
# We calculate a 90-day Standardized Precipitation Proxy from NASA rain data
master['Precip_90day'] = master.groupby('District')['LST'].transform(lambda x: x.rolling(window=90).sum())
master['SPI_Proxy'] = master.groupby('District')['Precip_90day'].transform(lambda x: (x - x.mean()) / x.std())

# 7. CROP PHENOLOGY MASKING (Q16 Logic)
master['Month'] = master['Date'].dt.month
# August-September (Kharif flowering) | January-February (Rabi flowering)
master['Is_Critical_Window'] = master['Month'].isin([8, 9, 1, 2]).astype(int)

# 8. THE FINAL COMPOUND STRESS INDEX (CSI) TRIGGER
master['CSI_Event'] = (master['Heat_Anomaly'] & master['Drought_Anomaly'] & master['Is_Critical_Window']).astype(int)

# 9. TARGET GENERATION (Ground Truth Validation for Q17)
# Failure = NDVI drops below the bottom 15% of historical health for that district
master['Target_Failure'] = (master['NDVI'] < master.groupby('District')['NDVI'].transform(lambda x: x.quantile(0.15))).astype(int)

# 10. SAVE PRODUCTION MASTER TABLE
os.makedirs("../data/processed", exist_ok=True)
master.to_csv("../data/processed/national_stack_production_table.csv", index=False)
print("✅ SUCCESS: 15-year Master Table created for MP!")
display(master.head()) 

Loading NASA Climate Backbone and GEE NDVI data...
Date Range fixed: 2010-01-01 00:00:00 to 2024-12-31 00:00:00
Merging layers into National Stack Master Table...
✅ Merge successful. Rows: 297024
Interpolating 16-day NDVI gaps to daily frequency...
Computing climatological anomalies and drought indices...
✅ SUCCESS: 15-year Master Table created for MP!


,Date,LST,SSM,District,NDVI,T_95th,SM_5th,Heat_Anomaly,Drought_Anomaly,Precip_90day,SPI_Proxy,Month,Is_Critical_Window,CSI_Event,Target_Failure
0,2010-01-01,25.31,0.62,AgarMalwa,NaN,43.6,0.59,0,0,NaN,NaN,1,1,0,0
1,2010-01-02,26.99,0.62,AgarMalwa,NaN,43.6,0.59,0,0,NaN,NaN,1,1,0,0
2,2010-01-03,27.35,0.62,AgarMalwa,NaN,43.6,0.59,0,0,NaN,NaN,1,1,0,0
3,2010-01-04,25.03,0.62,AgarMalwa,NaN,43.6,0.59,0,0,NaN,NaN,1,1,0,0
4,2010-01-05,24.80,0.62,AgarMalwa,NaN,43.6,0.59,0,0,NaN,NaN,1,1,0,0
